In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval av
except Exception:
    pass

In [2]:
import tempfile
from collections.abc import Generator
from pathlib import Path
from typing import NamedTuple, cast

import av
import numpy as np

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.data import SourceLocator
from dataeval.protocols import DatasetMetadata, DatumMetadata, MultiobjectTrackingDatum, VideoFrame
from dataeval.quality import Outliers

set_max_processes(1)

In [3]:
HEIGHT, WIDTH, FPS, N_FRAMES = 120, 160, 15, 30


def write_clip(path: Path, seed: int, flash: int | None = None) -> np.ndarray:
    """Encode one clip and return the box of its single track in every frame."""
    rng = np.random.default_rng(seed)
    background = rng.integers(60, 140, (HEIGHT // 8, WIDTH // 8, 3)).astype(np.uint8).repeat(8, 0).repeat(8, 1)
    boxes = []
    with av.open(str(path), "w") as container:
        stream = container.add_stream("mpeg4", rate=FPS)
        stream.width, stream.height, stream.pix_fmt = WIDTH, HEIGHT, "yuv420p"
        for i in range(N_FRAMES):
            image = background.copy()
            x = 10 + 4 * i
            image[40:70, x : x + 30] = 255
            if i == flash:
                image[:] = 250
            boxes.append([x, 40, x + 30, 70])
            for packet in stream.encode(av.VideoFrame.from_ndarray(image, format="rgb24")):
                container.mux(packet)
        for packet in stream.encode():
            container.mux(packet)
    return np.array(boxes, dtype=np.float32)


folder = Path(tempfile.mkdtemp())
paths = [folder / f"clip_{k}.mp4" for k in range(3)]
boxes = [write_clip(path, seed=k, flash=17 if k == 1 else None) for k, path in enumerate(paths)]
print([path.name for path in paths])

['clip_0.mp4', 'clip_1.mp4', 'clip_2.mp4']


In [4]:
class FrameTargets(NamedTuple):
    """The boxes in one frame, which MAITE calls a `SingleFrameObjectTrackingTarget`."""

    boxes: np.ndarray
    labels: np.ndarray
    scores: np.ndarray
    track_ids: np.ndarray


class VideoTargets(NamedTuple):
    """The boxes in every frame of one video, which MAITE calls a `MultiobjectTrackingTarget`."""

    frame_tracks: list[FrameTargets]


class Frame(NamedTuple):
    """One decoded frame, which MAITE calls a `VideoFrame`."""

    pixels: np.ndarray
    time_s: float
    pts: int
    frame_index: int


class FileStream:
    """The frames of one video file, decoded as they are read."""

    open_files = 0

    def __init__(self, path: Path) -> None:
        self.path = path

    def __iter__(self) -> Generator[VideoFrame, None, None]:
        container = av.open(str(self.path))
        FileStream.open_files += 1
        try:
            stream = container.streams.video[0]
            for index, frame in enumerate(container.decode(stream)):
                yield Frame(
                    pixels=frame.to_ndarray(format="rgb24").transpose(2, 0, 1),
                    time_s=float(frame.pts * stream.time_base),
                    pts=int(frame.pts),
                    frame_index=index,
                )
        finally:
            container.close()
            FileStream.open_files -= 1

In [5]:
class VideoFileDataset:
    """A MAITE multi-object tracking dataset over video files."""

    def __init__(self, paths: list[Path], boxes: list[np.ndarray], frame_rates: list[float] | None = None) -> None:
        self.metadata = DatasetMetadata({"id": "video-files", "index2label": {0: "car"}})
        self._paths, self._boxes, self._frame_rates = paths, boxes, frame_rates

    def __len__(self) -> int:
        return len(self._paths)

    def __getitem__(self, index: int) -> MultiobjectTrackingDatum:
        path = self._paths[index]
        with av.open(str(path)) as container:
            video = container.streams.video[0]
            info = {
                "id": path.stem,
                "frame_rate": float(video.average_rate),
                "video_duration": float(container.duration / av.time_base),
                "codec_name": video.codec_context.name,
            }
        if self._frame_rates is not None:
            info["frame_rate"] = self._frame_rates[index]
        frame_tracks = [
            FrameTargets(
                boxes=box[None],
                labels=np.array([0], dtype=np.int64),
                scores=np.ones(1, dtype=np.float32),
                track_ids=np.array([0], dtype=np.int64),
            )
            for box in self._boxes[index]
        ]
        return FileStream(path), VideoTargets(frame_tracks=frame_tracks), cast(DatumMetadata, info)


dataset = VideoFileDataset(paths, boxes)
metadata = Metadata(dataset)
print(metadata.rows_at("sequence").select("item_index", "frame_rate", "video_duration", "codec_name"))

shape: (3, 4)
┌────────────┬────────────┬────────────────┬────────────┐
│ item_index ┆ frame_rate ┆ video_duration ┆ codec_name │
│ ---        ┆ ---        ┆ ---            ┆ ---        │
│ i64        ┆ i64        ┆ i64            ┆ str        │
╞════════════╪════════════╪════════════════╪════════════╡
│ 0          ┆ 15         ┆ 2              ┆ mpeg4      │
│ 1          ┆ 15         ┆ 2              ┆ mpeg4      │
│ 2          ┆ 15         ┆ 2              ┆ mpeg4      │
└────────────┴────────────┴────────────────┴────────────┘


/tmp/ipykernel_6098/3204774288.py:37: UserWarning: `pts` and `time_s` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"pts": [...]} to control this.
  print(metadata.rows_at("sequence").select("item_index", "frame_rate", "video_duration", "codec_name"))


In [6]:
broken = Metadata(VideoFileDataset(paths, boxes, frame_rates=[-1.0, 15.0, 15.0]))
print(broken.dropped_factors)
print(broken.unusable_rows("frame_rate").sort("value").head(1))

{'frame_rate': ['invalid_known_factor']}
shape: (1, 3)
┌────────────┬───────┬─────────┐
│ item_index ┆ value ┆ kind    │
│ ---        ┆ ---   ┆ ---     │
│ i64        ┆ str   ┆ str     │
╞════════════╪═══════╪═════════╡
│ 0          ┆ -1    ┆ numeric │
└────────────┴───────┴─────────┘


In [7]:
result = Outliers().evaluate(dataset)
frames = result.frames.outliers
address = max(frames, key=lambda a: len(frames[a]))
found = SourceLocator(dataset)[address]
print(f"{address!r} is frame {address.key} of {paths[address.item].name}")
print(f"mean brightness {found.pixels.mean():.0f} of 255, flagged on {sorted(frames[address])}")

SourceIndex(1, 17, 'unit') is frame 17 of clip_1.mp4
mean brightness 248 of 255, flagged on ['brightness', 'contrast', 'darkness', 'entropy', 'kurtosis', 'mean', 'sharpness', 'skew', 'std', 'var']


In [8]:
# TEST ASSERTION CELL ###
first, second = list(FileStream(paths[0])), list(FileStream(paths[0]))
assert len(first) == N_FRAMES
assert [f.frame_index for f in first] == [f.frame_index for f in second] == list(range(N_FRAMES))
assert first[1].time_s > first[0].time_s
assert first[1].pts > first[0].pts
held = FileStream.open_files
walk = iter(FileStream(paths[0]))
next(walk)
assert FileStream.open_files == held + 1
walk.close()
assert FileStream.open_files == held
assert metadata.dropped_factors == {}
assert metadata.rows_at("sequence")["codec_name"].to_list() == ["mpeg4"] * 3
assert broken.dropped_factors == {"frame_rate": ["invalid_known_factor"]}
assert broken.unusable_rows("frame_rate").sort("value")["item_index"][0] == 0
assert (address.item, address.key) == (1, 17)
assert found.pixels.shape == (3, HEIGHT, WIDTH)
assert found.pixels.mean() > 240